# 1 Imports

In [1]:
import html
import os
import re
import pandas as pd
import numpy as np
import zipfile
import requests
from io import BytesIO
from tqdm import tqdm

In [2]:
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

# 2 Data loading

In [3]:
# tennis-data.co.uk moved its files into this folder (old http://tennis-data.co.uk/<year>/ links now return 404)
base = 'http://www.tennis-data.co.uk/hrjk-85HytOjkhth76j_ygh4jf7'
links = [f'{base}/{y}/{y}.{"xls" if y <= 2012 else "xlsx"}' for y in range(2000, 2027)]

## 2.1 Merge files into one dataframe

In [4]:
df = pd.DataFrame()
for i, elem in enumerate(links):
    # Download with requests: the site answers 403 to urllib, which pd.read_excel(url) uses
    content = requests.get(elem)
    content.raise_for_status()
    if elem[-4:] == '.zip':
        zf = zipfile.ZipFile(BytesIO(content.content))
        temp = pd.read_excel(zf.open(zf.namelist()[0])) 
    else:
        temp = pd.read_excel(BytesIO(content.content)) 
    df = pd.concat([df, temp], ignore_index=True)

/home/francisco/.pyenv/versions/3.11.8/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


/home/francisco/.pyenv/versions/3.11.8/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


/home/francisco/.pyenv/versions/3.11.8/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


/home/francisco/.pyenv/versions/3.11.8/lib/python3.11/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


# 3 Data preprocessing

In [5]:
df['Best of'] = df['Best of'].fillna(3)
# Retirements and walkovers are kept too (scored '... RET' / 'W/O' like Sackmann's data): they are the injury signal
df = df[df['Comment'].isin(['Completed', 'Retired', 'Walkover'])].reset_index(drop=True)
df = df[~df['WRank'].isnull()].reset_index(drop=True)
df = df[~df['LRank'].isnull()].reset_index(drop=True)
completed = df['Comment'] == 'Completed'
df = df[~completed | df[['W1', 'W2', 'L1', 'L2']].notna().all(axis=1)].reset_index(drop=True)

df['ind'] = [(lambda x: x % 2)(x) for x in range(len(df))]

# Unplayed sets come as NaN, ' ' or '\t' (2026 file), missing odds sometimes as '-'
sets = ['W1', 'L1', 'W2', 'L2', 'W3', 'L3', 'W4', 'L4', 'W5', 'L5']
df[sets] = df[sets].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)
df[['B365W', 'B365L']] = df[['B365W', 'B365L']].apply(pd.to_numeric, errors='coerce')

df['Player_1'] = df.apply(lambda row: row['Winner'] if row['ind'] == 0 else row['Loser'], axis = 1)
df['Player_2'] = df.apply(lambda row: row['Winner'] if row['ind'] == 1 else row['Loser'], axis = 1)
df['Rank_1'] = df.apply(lambda row: row['WRank'] if row['ind'] == 0 else row['LRank'], axis = 1)
df['Rank_2'] = df.apply(lambda row: row['WRank'] if row['ind'] == 1 else row['LRank'], axis = 1)
df['Pts_1'] = df.apply(lambda row: row['WPts'] if row['ind'] == 0 else row['LPts'], axis = 1)
df['Pts_2'] = df.apply(lambda row: row['WPts'] if row['ind'] == 1 else row['LPts'], axis = 1)
df['Odd_1'] = df.apply(lambda row: row['B365W'] if row['ind'] == 0 else row['B365L'], axis = 1)
df['Odd_2'] = df.apply(lambda row: row['B365W'] if row['ind'] == 1 else row['B365L'], axis = 1)

# Sharper / cheaper odds too (Pinnacle, best price across bookmakers, Betfair Exchange), same order as Odd_1/Odd_2
for name, w_col, l_col in (('PS', 'PSW', 'PSL'), ('Max', 'MaxW', 'MaxL'), ('BFE', 'BFEW', 'BFEL')):
    ow, ol = pd.to_numeric(df[w_col], errors='coerce'), pd.to_numeric(df[l_col], errors='coerce')
    df[f'{name}_1'] = np.where(df['ind'] == 0, ow, ol)
    df[f'{name}_2'] = np.where(df['ind'] == 1, ow, ol)

def score(df):
    if df['ind'] == 0:
        return str(df['W1']) + '-' + str(df['L1']) + ' ' + str(df['W2']) + '-' + str(df['L2']) + ' ' + str(df['W3']) + '-' + str(df['L3']) +\
 ' ' + str(df['W4']) + '-' + str(df['L4']) + ' ' + str(df['W5']) + '-' + str(df['L5']) + ' '
    return str(df['L1']) + '-' + str(df['W1']) + ' ' + str(df['L2']) + '-' + str(df['W2']) + ' ' + str(df['L3']) + '-' + str(df['W3']) +\
 ' ' + str(df['L4']) + '-' + str(df['W4']) + ' ' + str(df['L5']) + '-' + str(df['W5']) + ' '

df['Score'] = df.apply(lambda row: score(row).replace('0-0', ''), axis = 1)

df['Score'] = df['Score'].apply(lambda x: x.strip())
df.loc[df['Comment'] == 'Retired', 'Score'] = (df['Score'] + ' RET').str.strip()
df.loc[df['Comment'] == 'Walkover', 'Score'] = 'W/O'

new_df = df[['Tournament', 'Date', 'Series', 'Court', 'Surface', 'Round', 'Best of', 'Player_1', 'Player_2','Winner', 'Rank_1', 'Rank_2', 'Pts_1', 'Pts_2', 'Odd_1', 'Odd_2', 'PS_1', 'PS_2', 'Max_1', 'Max_2', 'BFE_1', 'BFE_2', 'Score']]

def check(str):
    return str.replace('NR', -1)

new_df['Date'] = pd.to_datetime(new_df['Date'], format = '%Y%M%d')
new_df = new_df.fillna(-1)
new_df[['Best of', 'Rank_1', 'Rank_2', 'Pts_1', 'Pts_2']] = new_df[['Best of', 'Rank_1', 'Rank_2', 'Pts_1', 'Pts_2']].apply(check)
new_df[['Best of', 'Rank_1', 'Rank_2', 'Pts_1', 'Pts_2']] = new_df[['Best of', 'Rank_1', 'Rank_2', 'Pts_1', 'Pts_2']].astype(float).astype(int)

/tmp/ipykernel_65581/2549672341.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['Date'] = pd.to_datetime(new_df['Date'], format = '%Y%M%d')
/tmp/ipykernel_65581/2549672341.py:47: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  return str.replace('NR', -1)


In [6]:
# Save in the schema data_extraction.ipynb reads (levels use Sackmann's codes: G, M, A, F = Tour Finals)
series_to_level = {
    'Grand Slam': 'G', 'Masters': 'M', 'Masters 1000': 'M', 'Masters Cup': 'F',
    'ATP500': 'A', 'ATP250': 'A', 'International': 'A', 'International Gold': 'A',
    'Championship Series': 'A', 'World Series': 'A',
}
new_df['tourney_level'] = new_df['Series'].map(series_to_level)
print("Unmapped Series:", new_df.loc[new_df['tourney_level'].isna(), 'Series'].unique())

new_df = new_df.drop(columns=['Series', 'Rank_1', 'Rank_2', 'Pts_1', 'Pts_2']).rename(columns={
    'Tournament': 'tourney_name', 'Date': 'tourney_date', 'Court': 'court', 'Surface': 'surface',
    'Round': 'round', 'Best of': 'best_of', 'Player_1': 'winner_name', 'Player_2': 'loser_name',
    'Winner': 'match_winner', 'Odd_1': 'winner_odds', 'Odd_2': 'loser_odds', 'Score': 'score',
})

os.makedirs('../data_set_3', exist_ok=True)
new_df.to_csv('../data_set_3/atp_tennis_matches.csv', index=False)

print("Latest date in the dataset:", new_df['tourney_date'].max())

Unmapped Series: []


Latest date in the dataset: 2026-10-18 00:00:00


# 4 Sackmann matches (data_set_2)

JeffSackmann/tennis_atp was taken off GitHub in Sept 2026. This archived mirror (snapshot of June 2026, CC BY-NC-SA 4.0, data by Jeff Sackmann) has tour-level matches until May 2026; tennis-data above covers the rest.

In [7]:
# Pinned to a commit so the mirror can't silently change under us
mirror = 'https://raw.githubusercontent.com/Aneeshers/tennis-sackmann-archive/83733587353df8a41f2fd4f516147d5aa83f5a8d/atp'
data_set_2 = '../data_set_2'
os.makedirs(data_set_2, exist_ok=True)

# All years from the mirror: the same matches as atp_matches_till_2022.csv from 1981 on (a few corrected scores),
# Sackmann's revised 1968-1980 data, and the per-match serve/return stats (from 1991) the old file had dropped
columns = ['tourney_name', 'surface', 'draw_size', 'tourney_level', 'tourney_date', 'winner_name', 'winner_ht', 'winner_age',
           'loser_name', 'loser_ht', 'loser_age', 'score', 'best_of', 'round', 'winner_name_clean', 'loser_name_clean']
stats = [f'{s}_{c}' for s in ('w', 'l') for c in ('1stIn', '1stWon', 'SvGms', 'bpSaved', 'bpFaced')]
sackmann = pd.concat([pd.read_csv(f'{mirror}/atp_matches_{y}.csv') for y in tqdm(range(1968, 2027))], ignore_index=True)
sackmann['winner_name_clean'] = sackmann['winner_name'].str.lower()
sackmann['loser_name_clean'] = sackmann['loser_name'].str.lower()
sackmann[columns + stats].to_csv(f'{data_set_2}/atp_matches_till_2026.csv', index=False)

# Newer player list (used by data_extraction to split first/last names)
pd.read_csv(f'{mirror}/atp_players.csv').to_csv(f'{data_set_2}/atp_players_till_2026.csv', index=False)

print(f"Sackmann matches: {len(sackmann)}, latest tournament start: {sackmann['tourney_date'].max()}")
print(f"with serve/return stats: {sackmann['w_1stIn'].notna().mean():.0%}")

  0%|          | 0/59 [00:00<?, ?it/s]

  2%|▏         | 1/59 [00:00<00:22,  2.63it/s]

  3%|▎         | 2/59 [00:00<00:18,  3.04it/s]

  5%|▌         | 3/59 [00:00<00:18,  3.09it/s]

  7%|▋         | 4/59 [00:01<00:18,  3.05it/s]

...

 98%|█████████▊| 58/59 [00:20<00:00,  2.63it/s]

100%|██████████| 59/59 [00:20<00:00,  2.81it/s]

100%|██████████| 59/59 [00:20<00:00,  2.89it/s]

Sackmann matches: 199389, latest tournament start: 20260525
with serve/return stats: 51%


/tmp/ipykernel_65581/768193865.py:17: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  pd.read_csv(f'{mirror}/atp_players.csv').to_csv(f'{data_set_2}/atp_players_till_2026.csv', index=False)


In [8]:
# Lower levels from the same archive, for the ratings in data_extraction (ratings.py): Challengers + tour qualifying
# (1978-2026) and Futures (1991-2026). The archive is pinned and never changes, so this only downloads once.
lower_file = f'{data_set_2}/atp_matches_lower_till_2026.csv'
if not os.path.exists(lower_file):
    keep = ['tourney_name', 'surface', 'tourney_level', 'tourney_date', 'round', 'winner_name', 'loser_name', 'score']
    def lower_year(kind, year):
        r = requests.get(f'{mirror}/atp_matches_{kind}_{year}.csv', timeout=120)
        return pd.read_csv(BytesIO(r.content), usecols=keep).assign(kind=kind) if r.ok else None
    jobs = [('qual_chall', y) for y in range(1978, 2027)] + [('futures', y) for y in range(1991, 2027)]
    pd.concat([lower_year(*job) for job in tqdm(jobs)], ignore_index=True).to_csv(lower_file, index=False)
lower = pd.read_csv(lower_file, low_memory=False)
print(f"Lower-level matches in the archive: {len(lower)} ({lower['kind'].value_counts().to_dict()}), up to {lower['tourney_date'].max()}")

Lower-level matches in the archive: 763826 ({'futures': 523355, 'qual_chall': 240471}), up to 20260601


# 5 Current ATP ranking (player search)

Every ranked player with name, country and birth date, from Tennis Abstract (updated every Monday). Only used to find players in predict_matches.py / predict_app.py, including those without tour-level matches yet; not used for training.

In [9]:
page = requests.get('https://www.tennisabstract.com/reports/atpRankings.html', timeout=60)
page.raise_for_status()
cells = [[html.unescape(re.sub(r'<[^>]+>', '', c)).replace('\xa0', ' ').strip() for c in re.findall(r'<td[^>]*>(.*?)</td>', row, re.S)]
         for row in re.findall(r'<tr[^>]*>(.*?)</tr>', page.text, re.S)]
ranking = pd.DataFrame([c for c in cells if len(c) == 4 and c[0].isdigit()], columns=['rank', 'name', 'country', 'dob'])
assert len(ranking) > 1000, f'Only {len(ranking)} players parsed: the ranking page layout may have changed'
ranking.to_csv(f'{data_set_2}/atp_current_ranking.csv', index=False)
print(f"Current ranking: {len(ranking)} players, {(ranking['dob'] != '').mean():.0%} with birth date")

Current ranking: 2296 players, 90% with birth date


# 6 Recent Challengers (Tennis Explorer)

Sackmann's Challenger files stop in June 2026, so the Challengers (main draw + qualifying) since then come from tennisexplorer.com (`tennis_explorer.py`). Finished events and known players are cached in `data_set_2/te_*.csv`: the first run takes about half an hour (one request per second), a weekly update a few minutes. Men's ITF/Futures results aren't on Tennis Explorer, so Futures stay at the archive (June 2026).

In [10]:
import tennis_explorer
from ratings import ARCHIVE_END

te_matches, te_players = tennis_explorer.update(f'{data_set_2}/te_challengers.csv', f'{data_set_2}/te_players.csv', since=ARCHIVE_END)
names = tennis_explorer.full_names(te_players, pd.read_csv(f'{data_set_2}/atp_players_till_2026.csv', low_memory=False), ranking)
recent = te_matches.assign(winner_name=te_matches['winner_slug'].map(names), loser_name=te_matches['loser_slug'].map(names),
                           tourney_level='C', kind='qual_chall').dropna(subset=['winner_name', 'loser_name']).astype({'tourney_date': int})
recent[['tourney_name', 'surface', 'tourney_level', 'tourney_date', 'round', 'winner_name', 'loser_name', 'score', 'kind']].to_csv(
    f'{data_set_2}/atp_matches_lower_recent.csv', index=False)
print(f"Challengers since {ARCHIVE_END}: {len(recent)} matches in {recent['tourney_name'].nunique()} events, up to {recent['tourney_date'].max()}; "
      f"{te_players['dob'].notna().mean():.0%} of {len(te_players)} players with birth date")

  events 5/5


Challengers since 20260601: 5276 matches in 110 events, up to 20261021.0; 88% of 1264 players with birth date
